In [3]:
import os
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel

In [9]:
load_dotenv(override=True)

# Load keys
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")
openrouter_key = os.getenv("OPENROUTER_API_KEY")

In [10]:
# Base URLs
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# 1. Tao async client cho moi provider
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_key)
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_key)

In [11]:
# 2. Tao model object
llama_model = OpenAIChatCompletionsModel(
    model="llama-3.3-70b-versatile",
    openai_client=groq_client
)

gemini_model = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=gemini_client
)

qwen_model = OpenAIChatCompletionsModel(
    model="qwen/qwen3.6-plus-preview:free",
    openai_client=openrouter_client
)

In [12]:
instructions_chuyen_nghiep = """Bạn là sales agent của GrowthVN.
Viết email chuyên nghiệp, tập trung ROI, tiếng Việt chuẩn mực."""

instructions_than_thien = """Bạn là sales agent của GrowthVN.
Viết email thân thiện, gần gũi với người đọc Việt Nam.
Dùng ngôn ngữ tự nhiên, đôi khi dí dỏm nhẹ nhàng."""

instructions_ngan_gon = """Bạn là sales agent của GrowthVN.
Viết email tối đa 5 câu, đi thẳng vào vấn đề. Không câu nào thừa."""

In [13]:
agent_llama = Agent(
    name="Llama Sales Agent",
    instructions=instructions_than_thien,
    model=llama_model
)

agent_gemini = Agent(
    name="Gemini Sales Agent",
    instructions=instructions_chuyen_nghiep,
    model=gemini_model
)

agent_qwen = Agent(
    name="Qwen Sales Agent",
    instructions=instructions_ngan_gon,
    model=qwen_model
)

In [7]:
# test nhanh
async def so_sanh_free_models():
    import time

    yeu_cau = "Viết email sales giới thiệu GrowthVN đến giám đốc SME Việt Nam"

    bat_dau = time.time()

    with trace("so-sanh-free-models"):
        ket_qua = await asyncio.gather(
            Runner.run(agent_llama, yeu_cau),
            Runner.run(agent_gemini, yeu_cau),
            Runner.run(agent_qwen, yeu_cau),
            return_exceptions=True
        )
    print(f"Hoàn thành trong {time.time() - bat_dau:.1f}s\n")

    labels = [
        "Llama 3.3 70B (Groq - free)",
        "Gemini 2.5 Flash (Google -free)",
        "Qwen (Openrouter - free)"
    ]

    for label, result in zip(labels, ket_qua):
        print(f"[{label}]")
        if isinstance(result, Exception):
            print(f"Lỗi: {result}")
        else:
            print(result.final_output)
        print()

await so_sanh_free_models()

Hoàn thành trong 19.5s

[Llama 3.3 70B (Groq - free)]
Chủ đề: Giải Pháp Tăng Trưởng Kinh Doanh Cho Doanh Nghiệp Của Bạn!

Kính gửi Anh/Chị [Tên Giám Đốc],

Hy vọng email này tìm đến Anh/Chị vào một ngày tốt lành và đầy năng lượng! Tôi là [Tên Sales Agent], đại diện của GrowthVN - Công ty chuyên về giải pháp tăng trưởng kinh doanh dành cho các doanh nghiệp vừa và nhỏ (SME) tại Việt Nam.

Tôi hiểu rằng, với vai trò là giám đốc của một doanh nghiệp SME, Anh/Chị luôn tìm kiếm những giải pháp hiệu quả để tăng trưởng và cạnh tranh trên thị trường. Tại GrowthVN, chúng tôi tự hào mang đến những dịch vụ và công cụ sáng tạo, giúp Anh/Chị đạt được mục tiêu kinh doanh của mình.

Dưới đây là một số lợi ích khi hợp tác với GrowthVN:

* **Tăng trưởng doanh thu**: Chúng tôi giúp doanh nghiệp của Anh/Chị tiếp cận với khách hàng tiềm năng mới và tăng cường mối quan hệ với khách hàng hiện tại.
* **Cải thiện hiệu suất**: Với các công cụ và kỹ thuật tiên tiến, chúng tôi giúp doanh nghiệp của Anh/Chị tinh g

# Structured Outputs voi Pydantic

In [1]:
from pydantic import BaseModel
from typing import Optional

class EmailOutput(BaseModel):
    tieu_de: str
    noi_dung: str
    phong_cach: str
    diem_thuyet_phuc: int
    ly_do: str            

In [4]:
agent_email_co_cau_truc = Agent(
    name="Structured Email Agent",
    instructions="""Bạn là sales agent của GrowthVN.
Viết email sales chuyên nghiệp cho doanh nghiệp Việt Nam.

Quan trọng: Bạn PHẢI trả về đúng cấu trúc được yêu cầu, bao gồm:
- tieu_de: tiêu đề email hấp dẫn, dưới 60 ký tự
- noi_dung: nội dung email đầy đủ, dưới 150 từ  
- phong_cach: mô tả phong cách viết bạn chọn
- diem_thuyet_phuc: điểm 1-10 bạn tự đánh giá email này
- ly_do: giải thích tại sao email này sẽ hiệu quả""",
    output_type=EmailOutput,
    model="gpt-4o-mini"
)


In [5]:
# chay va dung ket qua co cau truc
async def viet_email_co_cau_truc():
    yeu_cau = "Viết email sales GrowthVN gửi đến giám đốc marketing cho công ty thương mại điện tử"

    with trace("viet-email-co-cau-truc"):
        result = await Runner.run(agent_email_co_cau_truc, yeu_cau)
    print(result.final_output)

await viet_email_co_cau_truc()

tieu_de='Giải pháp tăng trưởng cho doanh nghiệp thương mại điện tử' noi_dung='Kính gửi Giám đốc Marketing,\n\nTại GrowthVN, chúng tôi hiểu rằng trong lĩnh vực thương mại điện tử, sự cạnh tranh luôn diễn ra gay gắt. Chúng tôi tự hào giới thiệu giải pháp tối ưu hóa SEO và quảng cáo trực tuyến giúp tăng trưởng doanh thu cho doanh nghiệp bạn.\n\nVới đội ngũ chuyên gia giàu kinh nghiệm, chúng tôi cam kết sẽ giúp bạn thu hút nhiều khách hàng hơn và nâng cao thương hiệu trên thị trường. \n\nHãy liên hệ với chúng tôi để khám phá cách chúng tôi có thể hỗ trợ bạn đạt được những mục tiêu kinh doanh của mình.\n\nTrân trọng,\n\n[Họ tên]\n[Chức vụ]\nGrowthVN\n[Thông tin liên hệ]' phong_cach='Chuyên nghiệp, gần gũi và thông tin ngắn gọn.' diem_thuyet_phuc=8 ly_do='Email ngắn gọn, rõ ràng và nêu bật được lợi ích mà dịch vụ của GrowthVN mang lại, từ đó thu hút sự chú ý của người nhận.'


# Guardrail

In [ ]:
# Dinh nghia schema cho Guardrail Agent
from pydantic import BaseModel

class KiemTraThongTinNhayCam(BaseModel):
    co_thong_tin_nhay_cam: bool
    loai_vi_pham: str
    noi_dung_vi_pham: str

In [7]:
# Tao guardrail Agent

guardrail_agent = Agent(
    name="Kiểm tra thông tin nhạy cảm",
    instructions="""Bạn kiểm tra xem yêu cầu của người dùng có chứa thông tin nhạy cảm không.

Các loại thông tin nhạy cảm cần phát hiện:
1. Tên người cụ thể (ví dụ: "gửi cho anh Minh", "từ chị Lan")
2. Số điện thoại (bất kỳ định dạng: 0912345678, +84912345678, 091-234-5678)
3. Email cá nhân cụ thể của người nhận

Nếu phát hiện → đặt co_thong_tin_nhay_cam = True và mô tả rõ vi phạm.
Nếu không có → đặt co_thong_tin_nhay_cam = False, loai_vi_pham = "khong_co".""",
    output_type=KiemTraThongTinNhayCam,
    model="gpt-4o-mini"
)

In [ ]:
# Guardrail Function
from agents import input_guardrail, GuardrailFunctionOutput

@input_guardrail # hoac @output_guardrail
async def bao_ve_thong_tin_nhay_cam(ctx, agent, message):

    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    kiem_tra: KiemTraThongTinNhayCam = result.final_output

    if kiem_tra.co_thong_tin_nhay_cam:
        print(f" Guardrail phát hiện: {kiem_tra.loai_vi_pham}")
        print(f" Chi tiết: {kiem_tra.noi_dung_vi_pham}")

    return GuardrailFunctionOutput(
        output_info={
            "loai_vi_pham": kiem_tra.loai_vi_pham,
            "chi_tiet": kiem_tra.noi_dung_vi_pham
        },
        tripwire_triggered=kiem_tra.co_thong_tin_nhay_cam
    )

In [14]:
# gan guardrail vao Sales Manager

# Tạo lại tool setup từ Day 2 
mo_ta = "Viết email sales giới thiệu dịch vụ GrowthVN"
tool1 = agent_llama.as_tool(tool_name="email_than_thien", tool_description=mo_ta)
tool2 = agent_gemini.as_tool(tool_name="email_chuyen_nghiep", tool_description=mo_ta)
tool3 = agent_qwen.as_tool(tool_name="email_ngan_gon", tool_description=mo_ta)

# Sales Manager co Guardrail
sales_manager_an_toan = Agent(
    name="Sales Manager An Toàn",
    instructions="""Bạn là Sales Manager của GrowthVN.
Dùng cả 3 tool để tạo 3 phiên bản email, chọn email phù hợp nhất.
Không tự viết email. Không tự gửi email.""",
    tools=[tool1, tool2, tool3],
    model="gpt-4o-mini",
    input_guardrails=[bao_ve_thong_tin_nhay_cam]
)

In [16]:
# test
async def test_guardrail():
    print("Test 1 - Yêu cầu có tên người:")
    try:
        with trace("test-guardrail-co-vi-pham"):
            result = await Runner.run(
                sales_manager_an_toan,
                "Gửi email sales đến anh Nguyễn Văn Minh, giám đốc công ty ABC"
            )
        print("Guardrail không hoạt động - đây là bug")
    except Exception as e:
        print(f"Guardrail hoạt động đúng: {type(e).__name__}")

    print("\n" + "-"*50 + "\n")

    print("Test 2 - Yêu cầu không có thông tin nhạy cảm:")
    try:
        with trace("test-guardrail-an-toan"):
            result = await Runner.run(
                sales_manager_an_toan,
                "Gửi email sales đến giám đốc marketing của các công ty SME Việt Nam"
            )
        print(f"Agent chạy thành công không vi phạm")
        print(f"output: {result.final_output[:100]}..")
    except Exception as e:
        print(f" Bị chặn nhầm: {e}")

await test_guardrail()
    

Test 1 - Yêu cầu có tên người:


 Guardrail phát hiện: Tên người cụ thể
 Chi tiết: Thông tin chứa tên cụ thể của cá nhân: 'anh Nguyễn Văn Minh'.
Guardrail hoạt động đúng: InputGuardrailTripwireTriggered

--------------------------------------------------

Test 2 - Yêu cầu không có thông tin nhạy cảm:
Agent chạy thành công không vi phạm
output: Dưới đây là ba phiên bản email giới thiệu dịch vụ của GrowthVN đến giám đốc marketing của các công t..
